# Baseline Model for Credit Card Fraud Detection (Kaggle PGS S6E11)

Welcome to the baseline notebook for **Kaggle Playground Series Season 6 Episode 11 (Predicting Credit Card Fraud)**.

### 🎯 Objectives
1. Establish a **leakage-free 5-fold Stratified Cross-Validation** pipeline (`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`).
2. Implement an interpretable, robust **Logistic Regression baseline** (`solver='newton-cholesky'`) with proper feature scaling and one-hot encoding.
3. Evaluate Out-Of-Fold (OOF) performance using **ROC-AUC** (the official competition metric) and **PR-AUC**.
4. Generate and verify the benchmark submission file (`baseline_submission.csv`).


In [1]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    classification_report
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries imported successfully!")


Libraries imported successfully!


## 1. Paths & Configuration

We set up flexible path detection so this notebook runs seamlessly both locally and inside Kaggle Kernels.


In [2]:
class Config:
    RANDOM_STATE = 42
    N_SPLITS = 5
    SOLVER = 'newton-cholesky'
    MAX_ITER = 500
    
    # Flexible path detection for Kaggle vs local execution
    if os.path.exists('/kaggle/input/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    elif os.path.exists('/kaggle/input/competitions/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    else:
        DATA_DIR = Path('../../dataset')
        if not (DATA_DIR / 'train.csv').exists():
            DATA_DIR = Path('pgs-s6e11/dataset')
        OUTPUT_DIR = Path('../../submission')
        if not OUTPUT_DIR.exists():
            OUTPUT_DIR = Path('pgs-s6e11/submission')

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using Data Directory: {Config.DATA_DIR}")
print(f"Using Output Directory: {Config.OUTPUT_DIR}")


Using Data Directory: ../../dataset
Using Output Directory: ../../submission


## 2. Data Ingestion & Sanity Checks

Let's load the training, test, and sample submission files and check the target class distribution.


In [3]:
train_df = pd.read_csv(Config.DATA_DIR / 'train.csv')
test_df = pd.read_csv(Config.DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(Config.DATA_DIR / 'sample_submission.csv')

print(f"Train Shape: {train_df.shape}")
print(f"Test Shape:  {test_df.shape}")
print(f"Sample Sub:  {sample_sub.shape}")

# Target class distribution
fraud_counts = train_df['is_fraud'].value_counts()
fraud_rates = train_df['is_fraud'].value_counts(normalize=True)
print("\n--- Target Distribution ('is_fraud') ---")
for label in [0.0, 1.0]:
    print(f"Class {int(label)}: {fraud_counts[label]:,d} ({fraud_rates[label]*100:.2f}%)")


Train Shape: (699960, 26)
Test Shape:  (299984, 25)
Sample Sub:  (299984, 2)

--- Target Distribution ('is_fraud') ---
Class 0: 688,271 (98.33%)
Class 1: 11,689 (1.67%)


## 3. Feature Categorization & Preprocessing Pipeline

We group features into:
- **Numerical Features**: Standard scaled (`StandardScaler`).
- **Categorical Features**: One-hot encoded (`OneHotEncoder`).
- **Boolean Features**: Cast to binary integers (0/1).


In [4]:
cat_cols = ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
bool_cols = [
    'is_foreign_transaction', 'is_new_merchant', 'used_vpn',
    'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt'
]

# Convert booleans to int in both train and test
for b in bool_cols:
    train_df[b] = train_df[b].astype(int)
    test_df[b] = test_df[b].astype(int)

# Numerical columns
num_cols = [c for c in train_df.columns if c not in cat_cols + ['id', 'is_fraud']]

print(f"Numerical features ({len(num_cols)}): {num_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary', sparse_output=False), cat_cols)
    ]
)


Numerical features (19): ['amount_usd', 'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'cvv_retry_count', 'velocity_score', 'time_of_day_hour', 'day_of_week', 'merchant_risk_score', 'prior_disputes', 'is_foreign_transaction', 'is_new_merchant', 'used_vpn', 'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt']
Categorical features (5): ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']


## 4. 5-Fold Stratified Cross-Validation

Using `StratifiedKFold` ensures every fold contains an identical ~1.67% proportion of fraud cases.
For each fold:
- We fit the preprocessor and Logistic Regression on the training fold.
- We predict out-of-fold validation probabilities to gauge generalization.
- We predict test set probabilities and accumulate fold-averaged predictions.


In [5]:
features = num_cols + cat_cols
X = train_df[features]
y = train_df['is_fraud'].astype(int)
X_test = test_df[features]

skf = StratifiedKFold(n_splits=Config.N_SPLITS, shuffle=True, random_state=Config.RANDOM_STATE)

oof_preds = np.zeros(len(X), dtype=np.float64)
test_preds = np.zeros(len(X_test), dtype=np.float64)
fold_scores = []

print("=" * 60)
print("Starting 5-Fold Stratified Cross-Validation (Logistic Regression)...")
print("=" * 60)

start_total = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    t0 = time.time()
    X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
    X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            solver=Config.SOLVER,
            max_iter=Config.MAX_ITER,
            random_state=Config.RANDOM_STATE
        ))
    ])
    
    pipeline.fit(X_train_fold, y_train_fold)
    
    val_probs = pipeline.predict_proba(X_val_fold)[:, 1]
    oof_preds[val_idx] = val_probs
    test_preds += pipeline.predict_proba(X_test)[:, 1] / Config.N_SPLITS
    
    fold_auc = roc_auc_score(y_val_fold, val_probs)
    fold_pr_auc = average_precision_score(y_val_fold, val_probs)
    fold_scores.append((fold_auc, fold_pr_auc))
    
    print(f"Fold {fold} finished in {time.time()-t0:.2f}s | ROC-AUC: {fold_auc:.5f} | PR-AUC: {fold_pr_auc:.5f}")

total_time = time.time() - start_total
print("=" * 60)
print(f"CV Complete in {total_time:.2f}s!")


Starting 5-Fold Stratified Cross-Validation (Logistic Regression)...
Fold 1 finished in 8.51s | ROC-AUC: 0.94126 | PR-AUC: 0.48214
Fold 2 finished in 7.43s | ROC-AUC: 0.94354 | PR-AUC: 0.49652
Fold 3 finished in 6.61s | ROC-AUC: 0.94687 | PR-AUC: 0.50317
Fold 4 finished in 7.47s | ROC-AUC: 0.94163 | PR-AUC: 0.48653
Fold 5 finished in 7.03s | ROC-AUC: 0.93633 | PR-AUC: 0.48112
CV Complete in 37.31s!


## 5. Model Evaluation & Performance Analysis

Let's compute the overall Out-Of-Fold ROC-AUC and PR-AUC scores and inspect diagnostic metrics.


In [6]:
overall_auc = roc_auc_score(y, oof_preds)
overall_pr_auc = average_precision_score(y, oof_preds)

print("-" * 50)
print("5-Fold Cross-Validation Summary:")
print(f"Mean Fold ROC-AUC:   {np.mean([s[0] for s in fold_scores]):.5f} ± {np.std([s[0] for s in fold_scores]):.5f}")
print(f"Overall OOF ROC-AUC: {overall_auc:.5f}")
print(f"Overall OOF PR-AUC:  {overall_pr_auc:.5f}")
print("-" * 50)

pred_binary = (oof_preds >= 0.5).astype(int)
print("\nClassification Report (threshold = 0.5):")
print(classification_report(y, pred_binary, digits=4))

# Plot ROC and PR Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr, tpr, _ = roc_curve(y, oof_preds)
axes[0].plot(fpr, tpr, color='crimson', lw=2, label=f'OOF ROC (AUC = {overall_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve (5-Fold OOF)')
axes[0].legend(loc='lower right')

precision, recall, _ = precision_recall_curve(y, oof_preds)
axes[1].plot(recall, precision, color='navy', lw=2, label=f'OOF PR (AUC = {overall_pr_auc:.4f})')
axes[1].axhline(y=y.mean(), color='gray', linestyle='--', label=f'No Skill Baseline ({y.mean():.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve (5-Fold OOF)')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.savefig('baseline_curves.png', dpi=150)
plt.show()


--------------------------------------------------
5-Fold Cross-Validation Summary:
Mean Fold ROC-AUC:   0.94193 ± 0.00343
Overall OOF ROC-AUC: 0.94190
Overall OOF PR-AUC:  0.48966
--------------------------------------------------

Classification Report (threshold = 0.5):
              precision    recall  f1-score   support

           0     0.9882    0.9983    0.9932    688271
           1     0.7423    0.2962    0.4234     11689

    accuracy                         0.9865    699960
   macro avg     0.8652    0.6472    0.7083    699960
weighted avg     0.9841    0.9865    0.9837    699960



## 6. Submission File Creation & Integrity Assertions

We output the final predictions to `baseline_submission.csv` and verify all integrity checks.


In [7]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'is_fraud': test_preds
})

assert submission.shape == sample_sub.shape, f"Shape mismatch: {submission.shape} vs {sample_sub.shape}"
assert list(submission.columns) == ['id', 'is_fraud'], f"Column mismatch: {submission.columns}"
assert not submission.isna().any().any(), "Submission contains null/NaN values!"
assert (submission['is_fraud'] >= 0.0).all() and (submission['is_fraud'] <= 1.0).all(), "Predictions out of [0, 1] range!"
assert (submission['id'] == sample_sub['id']).all(), "ID alignment mismatch!"

sub_path = Config.OUTPUT_DIR / 'baseline_submission.csv'
submission.to_csv(sub_path, index=False)
submission.to_csv(Config.OUTPUT_DIR / 'submission.csv', index=False)

print(f"Submission successfully verified and saved to: {sub_path}")
print(f"File size: {os.path.getsize(sub_path) / (1024*1024):.2f} MB")
print("\nFirst 10 submission rows:")
print(submission.head(10))


Submission successfully verified and saved to: ../../submission/baseline_submission.csv
File size: 8.15 MB

First 10 submission rows:
       id  is_fraud
0  699960  0.000291
1  699961  0.000169
2  699962  0.087717
3  699963  0.000845
4  699964  0.001669
5  699965  0.000388
6  699966  0.001929
7  699967  0.000217
8  699968  0.004419
9  699969  0.001507
